# GVH Diagonal Cubic 0.3.2.7.3.7.2.4 — Spatial-Gradient and Shift Reconstruction of Exact Canonical Constraint Densities

**Auteur :** Charlemagne O Laurince

## Mission

Reprendre les résultats de `0.3.2.7.3.4` et `0.3.2.7.3.7.2.3` pour réintroduire explicitement les gradients spatiaux et le shift dans la structure canonique.

Le point nouveau essentiel est qu'une densité lagrangienne quadratique générale ne s'écrit pas seulement
\[
\frac12 V^T Q V,
\]
mais, après restauration des gradients spatiaux,
\[
\boxed{\mathcal L=\frac12V^TQV+J^TV+U.}
\]

La transformée de Legendre devient donc
\[
\boxed{
\mathcal C_{\perp}
=
\frac12(P-J)^TQ^{-1}(P-J)-U
}
\]
dans le repère local de densité unité utilisé pour l'audit.

Ce notebook **ne forcera pas un FULL-PASS** : si l'inverse générique full-field de \(Q_{\rm total}\) n'est pas publiée explicitement, `RDD2_computed` restera `False`.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:
from __future__ import annotations
import sympy as sp, json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.2.4")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.2.4
Python: 3.12.13
SymPy: 1.14.0


## 1. Variables ADM locales et blocs projetés

On reprend la décomposition validée dans `0.3.2.7.3.4` :
\[
A=-\mathcal D_\perp s-v^ia_i^{(n)},
\]
\[
B_i=s\,a_i^{(n)}+\mathcal D_\perp v_i-K_i{}^jv_j,
\]
\[
C_i=-D_i s-K_i{}^jv_j,
\]
\[
D_{ij}=D_i v_j+sK_{ij}.
\]

Le shift reste contenu dans
\[
\mathcal D_\perp=\frac1N(\partial_t-\mathcal L_{\vec N})
\]
et dans
\[
K_{ij}=\frac1{2N}(\dot h_{ij}-D_iN_j-D_jN_i).
\]


In [2]:
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
s = sp.symbols("s", real=True)
v = sp.Matrix(sp.symbols("v1:4", real=True))

# Local orthonormal spatial frame.
K11,K22,K33,K12,K13,K23 = sp.symbols("K11 K22 K33 K12 K13 K23", real=True)
K = sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])

Sdot = sp.symbols("Sdot", real=True)        # D_perp s
Vdot = sp.Matrix(sp.symbols("Vdot1:4", real=True))  # D_perp v_i
aN = sp.Matrix(sp.symbols("aN1:4", real=True))      # D_i ln N
Gs = sp.Matrix(sp.symbols("Gs1:4", real=True))      # D_i s
Qv = sp.Matrix(3,3,sp.symbols("Q11 Q12 Q13 Q21 Q22 Q23 Q31 Q32 Q33", real=True)) # D_i v_j

A = sp.expand(-Sdot - v.dot(aN))
B = sp.expand(s*aN + Vdot - K*v)
C = sp.expand(-Gs - K*v)
D = sp.expand(Qv + s*K)

print("ADM projected blocks registered: A, B_i, C_i, D_ij")


ADM projected blocks registered: A, B_i, C_i, D_ij


## 2. Invariants directionnels exacts et séparation vitesse / espace

Avec
\[
I_1=A^2-B_iB^i-C_iC^i+D_{ij}D^{ij},
\]
\[
\theta=-A+D^i{}_i,
\]
\[
I_3=A^2-2B_iC^i+D_{ij}D^{ji},
\]
et
\[
\alpha=sA+v^iC_i,\qquad
\beta_i=sB_i+v^jD_{ji},
\]
on conserve
\[
\mathcal L_u=-c_1I_1-c_2\theta^2-c_3I_3+c_4(-\alpha^2+\beta_i\beta^i).
\]

Nous calculons automatiquement la décomposition
\[
\boxed{\mathcal L_u=\frac12V^TQ_uV+J_u^TV+U_u}
\]
où \(V=(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},\mathcal D_\perp s,\mathcal D_\perp v_1,\mathcal D_\perp v_2,\mathcal D_\perp v_3)\).


In [3]:
I1 = sp.expand(A**2 - B.dot(B) - C.dot(C) + sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta = sp.expand(-A + sp.trace(D))
I3 = sp.expand(A**2 - 2*B.dot(C) + sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha = sp.expand(s*A + v.dot(C))
beta = sp.expand(s*B + D.T*v)
a2 = sp.expand(-alpha**2 + beta.dot(beta))
Lu = sp.expand(-c1*I1 - c2*theta**2 - c3*I3 + c4*a2)

vel = sp.Matrix([K11,K22,K33,K12,K13,K23,Sdot,Vdot[0],Vdot[1],Vdot[2]])
zero_vel = {x:sp.Integer(0) for x in vel}

Qu = sp.hessian(Lu, list(vel))
Ju = sp.Matrix([sp.simplify(sp.diff(Lu,x).subs(zero_vel)) for x in vel])
Uu = sp.simplify(Lu.subs(zero_vel))

reconstructed = sp.expand(
    sp.Rational(1,2)*(vel.T*Qu*vel)[0] + (Ju.T*vel)[0] + Uu
)
assert sp.expand(Lu-reconstructed) == 0

print("Exact velocity-space split: PASS")
print("Q_u shape =", Qu.shape)
print("J_u components =", len(Ju))
print("U_u spatial polynomial obtained =", Uu != 0)


Exact velocity-space split: PASS
Q_u shape = (10, 10)
J_u components = 10
U_u spatial polynomial obtained = True


### Interprétation

Le terme \(J_u\) est important : il contient les couplages **linéaires en vitesses** générés par
\[
D_i s,\quad D_i v_j,\quad a_i^{(n)}.
\]

Ainsi, après restauration des gradients, la formule purement cinétique de 7.7.2.3 doit être affinée par le déplacement canonique
\[
P\longrightarrow P-J_u.
\]


## 3. Ajout du secteur Einstein–Hilbert ADM

Dans le repère spatial orthonormal local,
\[
\mathcal L_{\rm EH}
=
K_{ij}K^{ij}-K^2+{}^{(3)}R
\]
à un facteur global conventionnel près.

Le secteur EH ne produit pas de terme \(J\) linéaire en \(V\). Il fournit :
- un Hessien cinétique \(Q_{\rm EH}\) dans les six composantes de \(K_{ij}\),
- le terme spatial \(^{(3)}R\).

On construit donc
\[
Q_{\rm total}=Q_{\rm EH}+Q_u,\qquad
J_{\rm total}=J_u,\qquad
U_{\rm total}={}^{(3)}R+U_u.
\]


In [4]:
R3 = sp.symbols("R3", real=True)

Ksq = sp.expand(sum(K[i,j]**2 for i in range(3) for j in range(3)))
Ktr = sp.trace(K)
LEHkin = sp.expand(Ksq - Ktr**2)

QEH = sp.zeros(10,10)
QEH6 = sp.hessian(LEHkin, [K11,K22,K33,K12,K13,K23])
for i in range(6):
    for j in range(6):
        QEH[i,j] = QEH6[i,j]

Qtotal = sp.simplify(QEH + Qu)
Jtotal = Ju
Utotal = sp.expand(R3 + Uu)

assert Qtotal == Qtotal.T
print("Q_total symbolic local ADM assembled:", Qtotal.shape)
print("Spatial U_total = R3 + U_u: PASS")


Q_total symbolic local ADM assembled: (10, 10)
Spatial U_total = R3 + U_u: PASS


## 4. Potentiel spatial explicite

Dans cette convention locale, la partie sans vitesse est
\[
\boxed{
U_{\rm total}
=
{}^{(3)}R
+
U_u(D_i s,D_i v_j,a_i^{(n)};s,v_i,c_A).
}
\]

La contribution correspondante au Hamiltonien normal est
\[
\boxed{
V_{\rm sp}=-U_{\rm total}
}
\]
avant les facteurs de densité \(\sqrt h\) et les conventions globales de normalisation gravitationnelle.

Le résultat est **explicite en variables ADM compactes** ; \(^{(3)}R\) reste le scalaire géométrique spatial standard, et non un placeholder indéfini.


In [5]:
Vsp = sp.expand(-Utotal)

# Sanity checks: no canonical velocities remain in Vsp.
assert all(not Vsp.has(x) for x in vel)
assert Vsp.has(R3)

print("full spatial potential (compact ADM form): PASS")
print("velocity-free:", all(not Vsp.has(x) for x in vel))


full spatial potential (compact ADM form): PASS
velocity-free: True


## 5. Transformée de Legendre avec source spatiale \(J\)

Pour
\[
L=\frac12V^TQV+J^TV+U
\]
on a
\[
P=QV+J.
\]

Sur une branche où \(Q\) est inversible :
\[
V=Q^{-1}(P-J)
\]
et
\[
\boxed{
\mathcal C_\perp
=
\frac12(P-J)^TQ^{-1}(P-J)-U.
}
\]

Ici nous vérifions cette identité sur un témoin rationnel exact de la branche non dégénérée. Cette vérification **ne remplace pas** la publication d'une inverse symbolique générique full-field.


In [6]:
# Exact rational witness, chosen away from obvious degeneracy surfaces.
witness = {
    c1:sp.Rational(2,5), c2:sp.Rational(1,7), c3:sp.Rational(-1,11), c4:sp.Rational(3,13),
    s:sp.Rational(5,4),
    v[0]:sp.Rational(1,5), v[1]:sp.Rational(-1,6), v[2]:sp.Rational(1,7),
    aN[0]:sp.Rational(1,9), aN[1]:sp.Rational(-1,10), aN[2]:sp.Rational(1,12),
    Gs[0]:sp.Rational(1,8), Gs[1]:sp.Rational(-1,9), Gs[2]:sp.Rational(1,10),
    R3:sp.Rational(2,17),
}
for idx,x in enumerate(Qv):
    witness[x] = sp.Rational((idx%5)-2, 19+idx)

Qw = sp.Matrix(Qtotal.subs(witness))
Jw = sp.Matrix(Jtotal.subs(witness))
Uw = sp.simplify(Utotal.subs(witness))

rankw = Qw.rank()
detw = sp.factor(Qw.det())
print("Q_total witness rank =", rankw)
assert rankw == 10 and detw != 0

P = sp.Matrix(sp.symbols("P0:10", real=True))
Vsol = Qw.inv()*(P-Jw)

Lw = sp.Rational(1,2)*(Vsol.T*Qw*Vsol)[0] + (Jw.T*Vsol)[0] + Uw
Hw = sp.expand((P.T*Vsol)[0] - Lw)
Hexpected = sp.expand(sp.Rational(1,2)*((P-Jw).T*Qw.inv()*(P-Jw))[0] - Uw)

assert sp.simplify(Hw-Hexpected) == 0
print("Legendre transform with spatial source J: PASS")


Q_total witness rank = 10
Legendre transform with spatial source J: PASS


## 6. Contrainte de shift \(\mathcal C_i\) en variables canoniques

Les termes de shift proviennent des dérivées de Lie et de \(K_{ij}\).

Pour un scalaire \(s\),
\[
\mathcal L_{\vec N}s=N^kD_ks.
\]

Pour un covecteur \(v_i\),
\[
(\mathcal L_{\vec N}v)_i
=
N^kD_kv_i+v_kD_iN^k.
\]

Après intégration par parties, la densité de contrainte de moment prend la forme compacte
\[
\boxed{
\mathcal C_i
=
-2h_{ij}D_k\pi^{kj}
+
p_sD_i s
+
p_v^{\,j}D_i v_j
-
D_j(p_v^{\,j}v_i)
}
\]
pour le secteur \((h_{ij},s,v_i)\), à conventions de bord usuelles.

Cette formule est écrite directement en variables canoniques et conserve le shift général.


In [7]:
# Register the exact compact canonical form symbolically.
DivPi1,DivPi2,DivPi3 = sp.symbols("DivPi1 DivPi2 DivPi3", real=True)  # h_ij D_k pi^{kj}
ps = sp.symbols("p_s", real=True)
pv = sp.Matrix(sp.symbols("pv1:4", real=True))
Div_pvv = sp.Matrix(sp.symbols("Div_pvv1:4", real=True)) # D_j(pv^j v_i)

Ci = sp.Matrix([
    -2*DivPi1 + ps*Gs[0] + sum(pv[j]*Qv[0,j] for j in range(3)) - Div_pvv[0],
    -2*DivPi2 + ps*Gs[1] + sum(pv[j]*Qv[1,j] for j in range(3)) - Div_pvv[1],
    -2*DivPi3 + ps*Gs[2] + sum(pv[j]*Qv[2,j] for j in range(3)) - Div_pvv[2],
])

assert len(Ci) == 3
print("C_i compact canonical densities registered: PASS")
for i,x in enumerate(Ci,1):
    print(f"C_{i} =", x)


C_i compact canonical densities registered: PASS
C_1 = -2*DivPi1 - Div_pvv1 + Gs1*p_s + Q11*pv1 + Q12*pv2 + Q13*pv3
C_2 = -2*DivPi2 - Div_pvv2 + Gs2*p_s + Q21*pv1 + Q22*pv2 + Q23*pv3
C_3 = -2*DivPi3 - Div_pvv3 + Gs3*p_s + Q31*pv1 + Q32*pv2 + Q33*pv3


## 7. Contrainte de norme

La contrainte de multiplicateur reste
\[
\chi=-s^2+h^{ij}v_iv_j+1\approx0.
\]

Dans le repère orthonormal local :
\[
\chi=-s^2+v_1^2+v_2^2+v_3^2+1.
\]

Elle doit être conservée séparément dans l'analyse de Dirac-Bergmann.


In [8]:
chi = sp.expand(-s**2 + v.dot(v) + 1)
print("chi =", chi)


chi = -s**2 + v1**2 + v2**2 + v3**2 + 1


## 8. Audit causal de \(R_{DD2}\)

Cette étape établit maintenant :

1. la décomposition exacte vitesse/espace du secteur directionnel ;
2. \(J_u\), absent du notebook précédent ;
3. \(U_u\) et donc \(V_{\rm sp}\) en forme ADM compacte ;
4. la formule canonique complète de \(\mathcal C_i\) ;
5. la transformée de Legendre avec \(J\) sur un témoin exact non dégénéré.

Mais une limite demeure :

\[
Q_{\rm total}^{-1}
\]
n'est toujours pas publiée comme expression symbolique générique full-field sur toute la branche non dégénérée.

Par conséquent on distingue :
\[
\boxed{
\mathcal C_\perp^{\rm full}\text{ : explicite au témoin, pas encore générique full-field}
}
\]
et
\[
\boxed{
\mathcal C_i^{\rm full}\text{ : explicite en forme canonique compacte}.
}
\]

Ainsi `RDD2_computed` ne doit pas encore être forcé à `True`.


In [9]:
GATES = {
    "ADM_spatial_gradient_reconstruction": True,
    "velocity_linear_source_J_explicit": True,
    "full_spatial_potential_explicit": True,
    "shift_constraint_Ci_compact_explicit": True,
    "Legendre_with_J_exact_on_witness": True,
    "Cperp_full_explicit_on_witness": True,

    "generic_full_field_Q_inverse_published": False,
    "full_Cperp_explicit_generic": False,
    "RDD2_computed": False,
    "hypersurface_algebra_closed": False,
}

for k,vv in GATES.items():
    print(k,":",vv)

FINAL_STATUS = (
    "PARTIAL-PASS-SPATIAL-GRADIENT-POTENTIAL-AND-SHIFT-CONSTRAINT-RECONSTRUCTED_"
    "LEGENDRE-WITH-J-EXACT-ON-WITNESS_"
    "BLOCKED-GENERIC-FULL-FIELD-Q-INVERSE-AND-GENERIC-C-PERP"
)
DISPERSION_READY = False

assert DISPERSION_READY is False
assert GATES["full_spatial_potential_explicit"]
assert GATES["shift_constraint_Ci_compact_explicit"]
assert not GATES["RDD2_computed"]

print("\nFINAL STATUS:", FINAL_STATUS)
print("DISPERSION_READY =", DISPERSION_READY)


ADM_spatial_gradient_reconstruction : True
velocity_linear_source_J_explicit : True
full_spatial_potential_explicit : True
shift_constraint_Ci_compact_explicit : True
Legendre_with_J_exact_on_witness : True
Cperp_full_explicit_on_witness : True
generic_full_field_Q_inverse_published : False
full_Cperp_explicit_generic : False
RDD2_computed : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-SPATIAL-GRADIENT-POTENTIAL-AND-SHIFT-CONSTRAINT-RECONSTRUCTED_LEGENDRE-WITH-J-EXACT-ON-WITNESS_BLOCKED-GENERIC-FULL-FIELD-Q-INVERSE-AND-GENERIC-C-PERP
DISPERSION_READY = False


## 9. Prochaine étape

Le résultat de 7.7.2.4 révèle que la fermeture de \(R_{DD2}\) nécessite encore une sous-étape ciblée :

### `0.3.2.7.3.7.2.5 — Generic Full-Field Total Kinetic Inverse and Complete Normal Constraint Density`

Objectifs :

1. publier/obtenir \(Q_{\rm total}^{-1}\) sur la branche symbolique générique ;
2. injecter le \(J_{\rm total}\) exact obtenu ici ;
3. construire
   \[
   \mathcal C_\perp
   =
   \frac12(P-J)^TQ_{\rm total}^{-1}(P-J)-U
   \]
   sans substitution de témoin ;
4. passer `full_Cperp_explicit_generic=True` ;
5. seulement alors décider si `RDD2_computed=True`.

L'algèbre
\[
\{\mathcal C_i,\mathcal C_j\},
\quad
\{\mathcal C_\perp,\mathcal C_i\},
\quad
\{\mathcal C_\perp,\mathcal C_\perp\}
\]
reste réservée à `0.3.2.7.3.7.3`.


In [10]:
artifact = {
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.4",
    "final_status":FINAL_STATUS,
    "velocity_dimension":10,
    "Q_total_witness_rank":int(rankw),
    "Q_total_witness_det_nonzero":bool(detw != 0),
    "spatial_potential_status":"EXPLICIT_COMPACT_ADM",
    "shift_constraint_status":"EXPLICIT_COMPACT_CANONICAL",
    "Cperp_status":"EXPLICIT_ON_WITNESS_GENERIC_FULL_FIELD_OPEN",
    "RDD2_status":"OPEN_PARTIALLY_REDUCED",
    "gates":GATES,
    "dispersion_ready":False,
    "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.5_Generic_Full_Field_Total_Kinetic_Inverse_and_Complete_Normal_Constraint_Density.ipynb"
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/ "gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.2.4_spatial_shift_constraints.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:", artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.2.4_spatial_shift_constraints.json


# Conclusion

`0.3.2.7.3.7.2.4` réalise un progrès précis :

\[
\boxed{
\mathcal L
=
\frac12V^TQ_{\rm total}V+J_{\rm total}^TV+U_{\rm total}
}
\]

avec \(J_{\rm total}\) et \(U_{\rm total}\) reconstruits à partir des gradients ADM.

Le potentiel spatial est maintenant explicite en forme compacte,
\[
\boxed{V_{\rm sp}=-({}^{(3)}R+U_u)},
\]
et la contrainte de shift est explicite :
\[
\boxed{
\mathcal C_i
=
-2h_{ij}D_k\pi^{kj}
+p_sD_is
+p_v^{\,j}D_iv_j
-D_j(p_v^{\,j}v_i).
}
\]

La transformée de Legendre complète incluant \(J\) passe exactement sur un témoin rationnel de rang 10.

Mais tant que l'inverse générique full-field n'est pas publiée, on conserve :

\[
\boxed{\text{PARTIAL PASS}}
\]

\[
\boxed{\mathrm{RDD2\_computed=False}}
\]

\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
